In [2]:
import pandas as pd

# =============================================
# 데이터 불러오기
# =============================================

# orders.csv 파일 불러오기
df = pd.read_csv('data/orders.csv')

# 잘 불러왔는지 확인
print('데이터 불러오기 완료!')
print('데이터 크기:', df.shape)
print(df.head())

# 결측값 확인
print('결측값 확인:')
print(df.isnull().sum())


# =============================================
# 메모리 최적화
# (데이터가 너무 크기 때문에 꼭 필요!)
# =============================================

# 최적화 전에 메모리 얼마나 쓰는지 먼저 확인
before = df.memory_usage(deep=True).sum() / 1024**2
print('최적화 전 메모리:', round(before, 1), 'MB')

# 숫자 범위가 작은 컬럼은 더 작은 타입으로 바꿔주기
# 예) 요일은 0~6밖에 없으니까 int64 쓸 필요 없이 int8로 충분
df['order_dow']              = df['order_dow'].astype('int8')
df['order_hour_of_day']      = df['order_hour_of_day'].astype('int8')
df['order_number']           = df['order_number'].astype('int16')
df['days_since_prior_order'] = df['days_since_prior_order'].astype('float32')

# 최적화 후 메모리 얼마나 줄었는지 확인
after = df.memory_usage(deep=True).sum() / 1024**2
print('최적화 후 메모리:', round(after, 1), 'MB')
print('줄어든 양:', round(before - after, 1), 'MB')


# =============================================
# Feature 1 : 유저별 총 주문 건수
# → 주문을 많이 한 유저일수록 재구매할 가능성이 높다고 생각함
# =============================================

# user_id 기준으로 묶어서 주문 횟수 세기
총주문건수 = df.groupby('user_id')['order_id'].count()

# 인덱스 초기화해서 보기 좋게 만들기
총주문건수 = 총주문건수.reset_index()

# 컬럼 이름 바꾸기
총주문건수.columns = ['user_id', 'total_orders']

# 잘 만들어졌는지 확인
print('유저별 총 주문 건수:')
print(총주문건수.head())
print('전체 유저 수:', len(총주문건수))


# =============================================
# Feature 2 : 유저별 평균 재구매 주기
# → EDA 에서 평균 11.1일 주기 확인했음
# → 첫 주문은 이전 주문이 없으니까 NaN → 빼고 계산
# =============================================

# NaN 인 행 제거 (= 첫 주문 제거)
재구매데이터 = df[df['days_since_prior_order'].notnull()]
print('첫 주문 제거 후 행 수:', len(재구매데이터))

# user_id 기준으로 묶어서 평균 계산
재구매빈도 = 재구매데이터.groupby('user_id')['days_since_prior_order'].mean()
재구매빈도 = 재구매빈도.reset_index()
재구매빈도.columns = ['user_id', 'avg_days_between_orders']

print('유저별 평균 재구매 주기:')
print(재구매빈도.head())


# =============================================
# Feature 3 : 유저별 선호 요일
# → EDA 에서 일요일/월요일에 주문 집중 확인했음
# → 유저마다 자주 주문하는 요일이 다르니까 최빈값으로 뽑기
# =============================================

# mode() = 가장 많이 나온 값 (최빈값)
# [0] 은 최빈값이 여러 개일 때 첫 번째 것만 쓰려고
가장많은요일 = df.groupby('user_id')['order_dow'].agg(lambda x: x.mode()[0])
가장많은요일 = 가장많은요일.reset_index()
가장많은요일.columns = ['user_id', 'favorite_dow']

print('유저별 선호 요일:')
print(가장많은요일.head())


# =============================================
# Feature 4 : 유저별 선호 시간대
# → EDA 에서 오전 10시 피크 확인했음
# → 유저마다 자주 주문하는 시간대가 다르니까 최빈값으로 뽑기
# =============================================

가장많은시간 = df.groupby('user_id')['order_hour_of_day'].agg(lambda x: x.mode()[0])
가장많은시간 = 가장많은시간.reset_index()
가장많은시간.columns = ['user_id', 'favorite_hour']

print('유저별 선호 시간대:')
print(가장많은시간.head())


# =============================================
# Feature 5 : 재구매 주기 규칙성 [추가 제안]
# → 주기가 일정한 유저일수록 다음 주문 시점 예측이 쉬움
# → 표준편차가 작을수록 규칙적으로 재구매하는 유저
# =============================================

주기규칙성 = 재구매데이터.groupby('user_id')['days_since_prior_order'].std()
주기규칙성 = 주기규칙성.reset_index()
주기규칙성.columns = ['user_id', 'std_days_between_orders']

print('유저별 재구매 주기 규칙성:')
print(주기규칙성.head())


# =============================================
# Feature 6 : 충성도 지표 [추가 제안]
# → order_number 가 높을수록 오래된 충성 고객
# → 충성 고객일수록 재구매 가능성이 높다고 판단
# =============================================

최근주문번호 = df.groupby('user_id')['order_number'].max()
최근주문번호 = 최근주문번호.reset_index()
최근주문번호.columns = ['user_id', 'max_order_number']

print('유저별 충성도 지표:')
print(최근주문번호.head())


# =============================================
# 전부 합치기
# → user_id 기준으로 하나씩 붙이기
# =============================================

user_features = 총주문건수.merge(재구매빈도,   on='user_id')
user_features = user_features.merge(가장많은요일, on='user_id')
user_features = user_features.merge(가장많은시간, on='user_id')
user_features = user_features.merge(주기규칙성,   on='user_id')
user_features = user_features.merge(최근주문번호, on='user_id')

# 합쳐진 결과 확인
print('합친 후 user_features:')
print(user_features.head(10))
print('크기:', user_features.shape)


# =============================================
# 합친 후 메모리 최적화
# =============================================

user_features['total_orders']            = user_features['total_orders'].astype('int16')
user_features['avg_days_between_orders'] = user_features['avg_days_between_orders'].astype('float32')
user_features['favorite_dow']            = user_features['favorite_dow'].astype('int8')
user_features['favorite_hour']           = user_features['favorite_hour'].astype('int8')
user_features['std_days_between_orders'] = user_features['std_days_between_orders'].astype('float32')
user_features['max_order_number']        = user_features['max_order_number'].astype('int16')

print('최종 메모리:', round(user_features.memory_usage(deep=True).sum() / 1024**2, 1), 'MB')


# =============================================
# CSV 저장
# =============================================

user_features.to_csv('../data/user_features.csv', index=False)
print('저장 완료! → user_features.csv')


데이터 불러오기 완료!
데이터 크기: (3421083, 7)
   order_id  user_id eval_set  order_number  order_dow  order_hour_of_day  \
0   2539329        1    prior             1          2                  8   
1   2398795        1    prior             2          3                  7   
2    473747        1    prior             3          3                 12   
3   2254736        1    prior             4          4                  7   
4    431534        1    prior             5          4                 15   

   days_since_prior_order  
0                     NaN  
1                    15.0  
2                    21.0  
3                    29.0  
4                    28.0  
결측값 확인:
order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64
최적화 전 메모리: 332.7 MB
최적화 후 메모리: 254.4 MB
줄어든 양: 78.3 MB
유저별 총 주문 건수:
   user_id  tot

OSError: Cannot save file into a non-existent directory: '..\data'